In [4]:
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import json

In [15]:
import json

In [16]:
with open("../data/sample_candidates.json", "r", encoding="utf-8") as f:
    candidates = json.load(f)

print("Total Candidates:", len(candidates))

Total Candidates: 50


In [17]:
print(type(candidates))
print(candidates[0]["candidate_id"])

<class 'list'>
CAND_0000001


In [5]:
model = SentenceTransformer("all-MiniLM-L6-v2")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [6]:
import sentence_transformers
print("sentence-transformers imported successfully")

sentence-transformers imported successfully


In [7]:
import json
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
from docx import Document

In [8]:
model = SentenceTransformer("all-MiniLM-L6-v2")
print("Model Loaded")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Model Loaded


In [9]:
doc = Document("../data/job_description.docx")

job_text = ""

for paragraph in doc.paragraphs:
    job_text += paragraph.text + "\n"

print(job_text[:500])

Job Description: Senior AI Engineer — Founding Team
Company: Redrob AI (Series A AI-native talent intelligence platform)
Location: Pune/Noida, India (Hybrid — flexible cadence) | Open to relocation candidates from Tier-1 Indian cities
Employment Type: Full-time
Experience Required: 5–9 years (see "what we mean by this" below)

Let's be honest about this role
We're going to write this JD differently from most. We're a Series A company that just raised our round and we're building a new AI Enginee


In [10]:
job_embedding = model.encode(job_text)

print(job_embedding.shape)

(384,)


In [12]:
def create_candidate_profile(candidate):

    profile = candidate["profile"]

    skills = ", ".join(skill["name"] for skill in candidate["skills"])

    education = "\n".join(
        f"{edu['degree']} in {edu['field_of_study']} from {edu['institution']}"
        for edu in candidate["education"]
    )

    career = "\n".join(
        f"{job['title']} at {job['company']}: {job['description']}"
        for job in candidate["career_history"]
    )

    candidate_text = f"""
Headline:
{profile['headline']}

Summary:
{profile['summary']}

Experience:
{profile['years_of_experience']} years

Current Role:
{profile['current_title']}

Skills:
{skills}

Education:
{education}

Career History:
{career}
"""

    return candidate_text

In [18]:
candidate_text = create_candidate_profile(candidates[0])

print(candidate_text[:500])


Headline:
Backend Engineer | SQL, Spark, Cloud

Summary:
Software / data professional with 6.9 years of experience building data pipelines, backend systems, and analytics infrastructure. I'm a backend/data hybrid — Spark, Airflow, SQL warehouses are home territory; I'm building competence on the ML side. My toolkit is solid on the data engineering side — Python, SQL, Spark, Airflow, warehouse design — and I've completed a couple of self-directed ML projects (Kaggle competitions, side projects f


In [19]:
candidate_embedding = model.encode(candidate_text)

print(candidate_embedding.shape)

(384,)


In [20]:
similarity = cosine_similarity(
    [job_embedding],
    [candidate_embedding]
)[0][0]

print("Similarity Score:", similarity)

Similarity Score: 0.57278764


In [21]:
required_skills = [
    "Python",
    "Embeddings",
    "Retrieval",
    "Ranking",
    "Milvus",
    "FAISS",
    "Qdrant",
    "Weaviate",
    "Pinecone",
    "Elasticsearch",
    "OpenSearch",
    "LLM",
    "LoRA",
    "PEFT",
    "NLP",
    "Machine Learning",
    "Evaluation",
    "NDCG",
    "MAP",
    "MRR"
]

In [22]:
def calculate_skill_score(candidate):

    candidate_skills = [
        skill["name"].lower()
        for skill in candidate["skills"]
    ]

    matched = 0

    matched_skills = []

    for skill in required_skills:
        if skill.lower() in candidate_skills:
            matched += 1
            matched_skills.append(skill)

    score = matched / len(required_skills)

    return score, matched_skills

In [23]:
skill_score, matched = calculate_skill_score(candidates[0])

print("Skill Score:", skill_score)
print("Matched Skills:", matched)

Skill Score: 0.15
Matched Skills: ['Milvus', 'LoRA', 'NLP']


In [24]:
SKILL_MAP = {
    "LLM": ["Fine-tuning LLMs", "LoRA", "PEFT"],
    "Machine Learning": ["PyTorch", "TensorFlow", "Scikit-learn"],
    "Embeddings": ["Sentence Transformers", "BERT", "E5", "BGE"],
    "Vector Database": ["Milvus", "FAISS", "Pinecone", "Weaviate", "Qdrant"]
}

In [25]:
def calculate_behavior_score(candidate):
    signals = candidate["redrob_signals"]

    score = 0

    # Open to work
    if signals["open_to_work_flag"]:
        score += 0.25

    # Recruiter response rate (0–1)
    score += 0.25 * signals["recruiter_response_rate"]

    # Interview completion rate (0–1)
    score += 0.25 * signals["interview_completion_rate"]

    # GitHub activity (0–100)
    github = max(0, signals["github_activity_score"]) / 100
    score += 0.25 * github

    return score

In [26]:
behavior_score = calculate_behavior_score(candidates[0])
print("Behavior Score:", round(behavior_score, 3))

Behavior Score: 0.535


In [28]:
semantic_score = similarity

In [30]:
experience_score = candidates[0]["profile"]["years_of_experience"] / 10
experience_score = min(experience_score, 1.0)

final_score = (
    0.50 * semantic_score +
    0.20 * skill_score +
    0.20 * behavior_score +
    0.10 * experience_score
)

print("Final Score:", round(final_score, 4))

Final Score: 0.4925


In [31]:
semantic_score = cosine_similarity(
    [job_embedding],
    [candidate_embedding]
)[0][0]

skill_score, matched_skills = calculate_skill_score(candidates[0])

behavior_score = calculate_behavior_score(candidates[0])

experience_score = min(
    candidates[0]["profile"]["years_of_experience"] / 10,
    1.0
)

In [33]:
final_score = (
    0.50 * semantic_score +
    0.20 * skill_score +
    0.20 * behavior_score +
    0.10 * experience_score
)

print(f"Semantic Score : {semantic_score:.4f}")
print(f"Skill Score    : {skill_score:.4f}")
print(f"Behavior Score : {behavior_score:.4f}")
print(f"Experience     : {experience_score:.4f}")
print(f"Final Score    : {final_score:.4f}")

Semantic Score : 0.5728
Skill Score    : 0.1500
Behavior Score : 0.5355
Experience     : 0.6900
Final Score    : 0.4925


In [35]:
for candidate in candidates:
   

SyntaxError: incomplete input (785472012.py, line 2)

In [36]:
results = []

In [37]:
results = []

for candidate in candidates:

    # Candidate profile
    candidate_text = create_candidate_profile(candidate)

    # Embedding
    candidate_embedding = model.encode(candidate_text)

    # Semantic similarity
    semantic_score = cosine_similarity(
        [job_embedding],
        [candidate_embedding]
    )[0][0]

    # Skill score
    skill_score, matched = calculate_skill_score(candidate)

    # Behavior score
    behavior_score = calculate_behavior_score(candidate)

    # Experience score
    experience_score = min(
        candidate["profile"]["years_of_experience"] / 10,
        1.0
    )

    # Final score
    final_score = (
        0.50 * semantic_score +
        0.20 * skill_score +
        0.20 * behavior_score +
        0.10 * experience_score
    )

    results.append({
        "candidate_id": candidate["candidate_id"],
        "final_score": final_score,
        "semantic_score": semantic_score,
        "skill_score": skill_score,
        "behavior_score": behavior_score,
        "experience_score": experience_score,
        "matched_skills": matched
    })

In [38]:
import pandas as pd

results_df = pd.DataFrame(results)

results_df.head()

,candidate_id,final_score,semantic_score,skill_score,behavior_score,experience_score,matched_skills
0,CAND_0000001,0.492494,0.572788,0.15,0.5355,0.69,"[Milvus, LoRA, NLP]"
1,CAND_0000002,0.497898,0.604795,0.00,0.4775,1.00,[]
2,CAND_0000003,0.368797,0.583594,0.00,0.3300,0.11,[]
3,CAND_0000004,0.355851,0.574702,0.00,0.1525,0.38,[]
4,CAND_0000005,0.467828,0.524655,0.00,0.5275,1.00,[]


In [39]:
results_df = results_df.sort_values(
    by="final_score",
    ascending=False
)

results_df.head(10)

,candidate_id,final_score,semantic_score,skill_score,behavior_score,experience_score,matched_skills
30,CAND_0000031,0.515522,0.547444,0.20,0.7090,0.60,"[Embeddings, FAISS, Pinecone, Machine Learning]"
20,CAND_0000021,0.502284,0.636168,0.15,0.2710,1.00,"[Embeddings, FAISS, Pinecone]"
1,CAND_0000002,0.497898,0.604795,0.00,0.4775,1.00,[]
37,CAND_0000038,0.492821,0.598842,0.05,0.5820,0.67,[Weaviate]
0,CAND_0000001,0.492494,0.572788,0.15,0.5355,0.69,"[Milvus, LoRA, NLP]"
44,CAND_0000045,0.491800,0.588601,0.00,0.4875,1.00,[]
35,CAND_0000036,0.490581,0.563161,0.00,0.5450,1.00,[]
24,CAND_0000025,0.489305,0.588609,0.00,0.6100,0.73,[]
40,CAND_0000041,0.486209,0.580418,0.00,0.4800,1.00,[]
47,CAND_0000048,0.481999,0.562997,0.00,0.5175,0.97,[]
